In [ ]:
import scipy.io
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
## if loading mat file.
mat=scipy.io.loadmat('/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplateMats/000129_21648_NetworkTemplates.mat')
print(mat)

In [ ]:
import os
import glob

# Specify the directory path where you want to search for .mat files
directory_path = '/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplateMats/'

# Use glob to find all .mat files in the directory
mat_files = glob.glob(os.path.join(directory_path, '*.mat'))

# mat_files will now contain a list of file paths to all the .mat files in the directory
print("List of .mat files:")
for mat_file in mat_files:
    print(mat_file)


In [ ]:
import os

file_path = '/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplateMats/000138_21661_NetworkTemplates.mat'

# Get the base name of the file (including extension)
file_name_with_extension = os.path.basename(file_path)

# Split the file name and extension
file_name, file_extension = os.path.splitext(file_name_with_extension)

# Now, file_name contains the name of the file without the extension
print("File name without extension:", file_name)

In [ ]:
from sklearn.cluster import KMeans
import pickle
def calculate_burst_features(stacked_bursts, threshold=0.5):

    feature_array = np.empty((len(stacked_bursts), 1)) 
    
    for i,burst in enumerate(stacked_bursts):
    
        # Find the peak amplitude and its index
        peak_amplitude = np.max(burst)
        peak_index = np.argmax(burst)
        # Calculate the half-maximum threshold
        half_threshold = threshold * peak_amplitude

        # Search for the left boundary where the amplitude first crosses half-maximum
        left_boundary = np.where(burst[:peak_index] <= half_threshold)[0]
        if len(left_boundary) == 0:
            left_boundary_index = 0
        else:
            left_boundary_index = left_boundary[-1] + 1

        # Search for the right boundary where the amplitude crosses half-maximum again
        right_boundary = np.where(burst[peak_index:] <= half_threshold)[0]
        if len(right_boundary) == 0:
            right_boundary_index = len(burst) - 1
        else:
            right_boundary_index = peak_index + right_boundary[0]

        # Calculate the half-width
        half_width = right_boundary_index - left_boundary_index + 1

        feature_array[i] = [peak_amplitude]
    

    return feature_array




for mat_file in mat_files:

    mat=scipy.io.loadmat(mat_file)
    data = mat['firingRateNorm'] 
    
    burst_start = False
    peak = 0
    peak_list=[]
    burst_start = False
    peak_index_list=[]
    peak_index = 0
    threshold = 3.5
    total = len(data)


    for x in range(1, total-2):
        if data[x] >= threshold and data[x] > data[x-1] and data[x] > data[x+1] and data[x] > peak:
            burst_start = True
            peak = data[x]
            peak_index = x
        if data[x] < 1 and burst_start == True:
            peak_list.append(peak)
            peak_index_list.append(peak_index)
            peak=0
            peak_index = 0
            burst_start = False  
        peak_index_list = [value for value in peak_index_list if value < 0 or value > 50]
        print(peak_index_list)
        #generating a list containing all the bursts/the values representing individual burst in the data starting from the index of the highest peaks.
        list_of_burst = []
        left_bound = 30
        right_bound =30
        for index in peak_index_list:
            individual_burst=np.array(data[index-left_bound:index+right_bound]).flatten()

            #individual_burst.sort()
            list_of_burst.append(individual_burst)

        # #the values were represented as element in Numpy array, which make it impossible to perform PCC. Hence, the following codes extract the values out
        # for i in range(0, len(list_of_burst)):
        #     for x in range(0, len(list_of_burst[0])):
        #         list_of_burst[i][x] = list_of_burst[i][x][0]
        stacked_bursts = np.vstack(list_of_burst)


        

        features = calculate_burst_features(stacked_bursts, threshold=0.5)

        # Assuming features is a NumPy array where each row represents a burst with two columns (half-width, peak)

        # Initialize an empty list to store the WCSS values for different values of K
        wcss = []

        # Define a range of values for K (number of clusters)
        k_values = range(1, 11)  # You can adjust the range as needed

        # Calculate WCSS for each value of K
        for k in k_values:
            kmeans = KMeans(n_clusters=k)
            kmeans.fit(features[:,0].reshape(-1,1))
            wcss.append(kmeans.inertia_)
        # Calculate the change in slope between consecutive WCSS values
        slope_changes = [abs(wcss[i] - wcss[i - 1] )for i in range(1, len(wcss))]

        # Find the index with the maximum slope change
        optimal_k_index = slope_changes.index(max(slope_changes)) + 1  # Adding 1 because of zero-based indexing

        # Optimal K is the K value at the inflection point with the maximum slope change
        optimal_k = k_values[optimal_k_index]

        kmeans = KMeans(n_clusters=optimal_k)
        cluster_labels = kmeans.fit_predict(features)
        # print(cluster_labels)
        # Now, cluster_labels contains the cluster assignments for each burst

        # Create a dictionary to store features and their indices for each cluster
        cluster_data = {cluster_id: {'features': [], 'indices': []} for cluster_id in range(optimal_k)}

        # Assign features and their indices to their respective clusters
        for i, label in enumerate(cluster_labels):
            cluster_data[label]['features'].append(features[i])
            cluster_data[label]['indices'].append(i)

        for label in cluster_data.keys():
            cluster_data[label]['template'] = np.mean(stacked_bursts[cluster_data[label]['indices'],:])
        # Access features and indices for a specific cluster (e.g., cluster 0)

        file_name_with_extension = os.path.basename(mat_file)

        # Split the file name and extension
        file_name, file_extension = os.path.splitext(file_name_with_extension)
        pickle.dump(cluster_data,f'./PickelFile/{file_name}.pkl')

In [ ]:
len(data)

In [ ]:
data = mat['firingRateNorm']  # visualizing the data
plt.figure(figsize=(10, 5))
plt.plot(data[4600:9600])
plt.xlabel('Time (5e-2 ms)')
plt.ylabel('kHz')
plt.title('Line Plot of Data')
plt.ylim([0 ,10])
plt.savefig('/home/mmp/Documents/Images_25sep/templatematching_signal_control.pdf',format='pdf')
plt.show()
print(mat.keys())
total = len(data)
print(total)

In [ ]:
#find the index of the highest point in each burst
burst_start = False
peak = 0
peak_list=[]
burst_start = False
peak_index_list=[]
peak_index = 0
threshold = 3


for x in range(1, total-2):
    if data[x] >= threshold and data[x] > data[x-1] and data[x] > data[x+1] and data[x] > peak:
        burst_start = True
        peak = data[x]
        peak_index = x
    if data[x] < 1 and burst_start == True:
        peak_list.append(peak)
        peak_index_list.append(peak_index)
        peak=0
        peak_index = 0
        burst_start = False  
peak_index_list = [value for value in peak_index_list if value < 0 or value > 50]
print(peak_index_list)

#skipping a little in the beginingn to get correct shape of the first peak


In [ ]:
#generating a list containing all the bursts/the values representing individual burst in the data starting from the index of the highest peaks.
list_of_burst = []
left_bound = 75
right_bound =75
for index in peak_index_list:
    individual_burst=np.array(data[index-left_bound:index+right_bound]).flatten()

    #individual_burst.sort()
    list_of_burst.append(individual_burst)

# #the values were represented as element in Numpy array, which make it impossible to perform PCC. Hence, the following codes extract the values out
# for i in range(0, len(list_of_burst)):
#     for x in range(0, len(list_of_burst[0])):
#         list_of_burst[i][x] = list_of_burst[i][x][0]
stacked_bursts = np.vstack(list_of_burst)
print(stacked_bursts.shape)

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have imported necessary libraries, and you have 'stacked_bursts' as a 2D NumPy array

# Plot all rows (bursts) on top of each other
plt.figure(figsize=(2,10))  # Adjust the figure size as needed
plt.clf()
for row in stacked_bursts:
    plt.plot(row,c='lightblue')
template = np.mean(stacked_bursts,axis=0)
plt.plot(template,c='blue')
plt.ylim([0,10])
plt.xlabel('Time (5e-2 ms)')
plt.ylabel('Amplitude (microV)')
plt.title('Stacked Bursts')
plt.savefig('/home/mmp/Documents/Images_25sep/templatematching_aggregate_control.pdf',format='pdf')
plt.show()

In [ ]:
template = np.mean(stacked_bursts,axis=0)
plt.figure(figsize=(2, 10)) 
plt.plot(template)

plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Stacked Bursts')

plt.show()
plt.clf()

In [ ]:
#whole routine for the biggest template.
import pandas as pd
import pickle
dataframe = pd.DataFrame(columns=["Filename","Stacked Bursts","Template"])
error_l=[]
for mat_file in mat_files:

    mat=scipy.io.loadmat(mat_file)
    data = mat['firingRateNorm'] 
    total = len(data)
    file_name_with_extension = os.path.basename(mat_file)

    # Split the file name and extension
    file_name, file_extension = os.path.splitext(file_name_with_extension)
    print(file_name)
   #find the index of the highest point in each burst
    burst_start = False
    peak = 0
    peak_list=[]
    burst_start = False
    peak_index_list=[]
    peak_index = 0
    threshold = 3.5
    

    for x in range(1, total-2):
        if data[x] >= threshold and data[x] > data[x-1] and data[x] > data[x+1] and data[x] > peak:
            burst_start = True
            peak = data[x]
            peak_index = x
        if data[x] < 1 and burst_start == True:
            peak_list.append(peak)
            peak_index_list.append(peak_index)
            peak=0
            peak_index = 0
            burst_start = False  
    peak_index_list = [value for value in peak_index_list if value < 0 or value > 50]
    if not peak_index_list:

        error_l.append(file_name)
        continue
        
    #generating a list containing all the bursts/the values representing individual burst in the data starting from the index of the highest peaks.
    list_of_burst = []
    left_bound = 50
    right_bound =50
    for index in peak_index_list:
        individual_burst=np.array(data[index-left_bound:index+right_bound]).flatten()

        #individual_burst.sort()
        list_of_burst.append(individual_burst)

        # #the values were represented as element in Numpy array, which make it impossible to perform PCC. Hence, the following codes extract the values out
        # for i in range(0, len(list_of_burst)):
        #     for x in range(0, len(list_of_burst[0])):
        #         list_of_burst[i][x] = list_of_burst[i][x][0]
    stacked_bursts = np.vstack(list_of_burst)

    template = np.mean(stacked_bursts,axis=0)
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/StackedBurstPickelFile/{file_name}.pkl",'wb') as f1:
        pickle.dump(stacked_bursts,f1)
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplatesPickelFile/{file_name}.pkl",'wb') as f2:
        pickle.dump(template,f2)
    new_data = {"Filename": file_name, "Stacked Bursts": stacked_bursts, "Template": template}
    dataframe = pd.concat([dataframe, pd.DataFrame([new_data])], ignore_index=True)



# Save the DataFrame to Excel
dataframe.to_excel("./template_outputs3.xlsx", index=False, engine="openpyxl")


In [ ]:
print(error_l)

In [ ]:
print(error_l)

In [ ]:
df1 = pd.read_excel("/mnt/disk15tb/mmpatil/MEA_Analysis/TemplateMatching/template_outputs.xlsx")

In [ ]:
# Define a function to assign values to the "line" column based on the "Chip_ID" values
def assign_line(chip_id):
    if chip_id in [21670, 21648, 21609, 21613]:
        return 'syngap control'
    elif chip_id in [21625, 21661, 21664, 21644]:
        return 'Syngap'
    else:
        return None

# Apply the function to create the "line" column
df1['line'] = df1['chip'].apply(assign_line)

In [ ]:
df1.head(8)

In [ ]:
df2 = pd.read_excel("/mnt/disk15tb/mmpatil/MEA_Analysis/TemplateMatching/template_outputs3.xlsx")

In [ ]:
df2.head()

In [ ]:
df2['DIV']=df1['DIV']
df2['chip'] = df1['chip']
df2['line']=df1['line']

In [ ]:
sorted_df = df2.sort_values(by='DIV')
display(sorted_df)

In [ ]:
condtion1=(sorted_df['DIV']==88)
condition2=(sorted_df['line']=='syngap control')

filtered_df = sorted_df[condtion1&condition2]



In [ ]:
template_list =[]
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplatesPickelFile/{file_name}.pkl",'rb') as f4:
        template_array= pickle.load(f4)
    template_list.append(template_array)
template_array = np.array(template_list)

mean_template_wt = np.mean(template_array,axis=0)


In [ ]:
stacked_bursts_array = []
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/StackedBurstPickelFile/{file_name}.pkl",'rb') as f5:
        stacked_bursts_here = pickle.load(f5)
        # if stacked_bursts_array is None:
        #     stacked_bursts_array  = stacked_bursts_here
        # else :
        stacked_bursts_array.append(stacked_bursts_here)
stacked_arry_wt = np.vstack(stacked_bursts_array)
stacked_arry_wt.shape

In [ ]:
pcc_scores_wt_wt= [np.corrcoef(mean_template_wt,burst)[0,1] for burst in stacked_arry_wt ]
print(pcc_scores_wt_wt)


In [ ]:
condtion1=(sorted_df['DIV']==88)
condition2=(sorted_df['line']=='Syngap')

filtered_df = sorted_df[condtion1&condition2]
template_list =[]
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplatesPickelFile/{file_name}.pkl",'rb') as f6:
        template_array= pickle.load(f6)
    template_list.append(template_array)
template_array = np.array(template_list)

mean_template_syngap = np.mean(template_array,axis=0)

stacked_bursts_array = []
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/StackedBurstPickelFile/{file_name}.pkl",'rb') as f7:
        stacked_bursts_here = pickle.load(f7)
        # if stacked_bursts_array is None:
        #     stacked_bursts_array  = stacked_bursts_here
        # else :
        stacked_bursts_array.append(stacked_bursts_here)
stacked_arry_syngap = np.vstack(stacked_bursts_array)
print(stacked_arry_syngap.shape)
pcc_scores_wt_syngap= [np.corrcoef(mean_template_wt,burst)[0,1] for burst in stacked_arry_syngap ]
print(pcc_scores_wt_syngap)

In [ ]:
condtion1=(sorted_df['DIV']==101)
condition2=(sorted_df['line']=='syngap control')
f6=None
f7=None
filtered_df = sorted_df[condtion1&condition2]
template_list =[]
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplatesPickelFile/{file_name}.pkl",'rb') as f6:
        template_array= pickle.load(f6)
    template_list.append(template_array)
template_array = np.array(template_list)

mean_template_wt_101 = np.mean(template_array,axis=0)

stacked_bursts_array = []
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/StackedBurstPickelFile/{file_name}.pkl",'rb') as f7:
        stacked_bursts_here = pickle.load(f7)
        # if stacked_bursts_array is None:
        #     stacked_bursts_array  = stacked_bursts_here
        # else :
        stacked_bursts_array.append(stacked_bursts_here)
stacked_arry_control_div101 = np.vstack(stacked_bursts_array)
print(stacked_arry_control_div101.shape)
pcc_scores_wt_wt_101= [np.corrcoef(mean_template_wt_101,burst)[0,1] for burst in stacked_arry_control_div101 ]
print(pcc_scores_wt_wt_101)

In [ ]:
condtion1=(sorted_df['DIV']==101)
condition2=(sorted_df['line']=='Syngap')
f6=None
f7=None
filtered_df = sorted_df[condtion1&condition2]
template_list =[]
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/TemplatesPickelFile/{file_name}.pkl",'rb') as f6:
        template_array= pickle.load(f6)
    template_list.append(template_array)
template_array = np.array(template_list)

mean_template_syngap_101 = np.mean(template_array,axis=0)

stacked_bursts_array = []
for i in range(len(filtered_df)):
    file_name = filtered_df['Filename'].iloc[i]
    with open(f"/home/mmp/disktb/mmpatil/MEA_Analysis/TemplateMatching/StackedBurstPickelFile/{file_name}.pkl",'rb') as f7:
        stacked_bursts_here = pickle.load(f7)
        # if stacked_bursts_array is None:
        #     stacked_bursts_array  = stacked_bursts_here
        # else :
        stacked_bursts_array.append(stacked_bursts_here)
stacked_arry_syngap_div101 = np.vstack(stacked_bursts_array)
print(stacked_arry_syngap_div101.shape)
pcc_scores_wt_syngap_101= [np.corrcoef(mean_template_wt_101,burst)[0,1] for burst in stacked_arry_syngap_div101 ]
print(pcc_scores_wt_syngap_101)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random


total_div =1
# Convert data lists to NumPy arrays
group1_data = np.array(pcc_scores_wt_wt)
group2_data = np.array(pcc_scores_wt_syngap)

# Calculate means and standard errors of means (SEM)
mean1 = np.mean(group1_data)
sem1 = np.std(group1_data) / np.sqrt(len(group1_data))
mean2 = np.mean(group2_data)
sem2 = np.std(group2_data) / np.sqrt(len(group2_data))

# Convert data lists to NumPy arrays
group3_data = np.array(pcc_scores_wt_wt_101)
group4_data = np.array(pcc_scores_wt_syngap_101)

# Calculate means and standard errors of means (SEM)
mean3 = np.mean(group3_data)
sem3 = np.std(group3_data) / np.sqrt(len(group3_data))
mean4 = np.mean(group4_data)
sem4 = np.std(group4_data) / np.sqrt(len(group4_data))

# Create a bar graph with two-color bars and jittered data points along the x-axis
bar_width = 0.05  # Width of each bar

x_wt = 1
x_het = 1.05



  # X-axis positions for two groups
opacity = 0.7
jitter = 0.02  # Amount of jitter along the x-axis



# Create bars for Group 1 with two colors
plt.bar(x_wt, mean1, bar_width, alpha=opacity, color='red', yerr=sem1, capsize=10,ecolor='black',label='Control')
plt.bar(x_het, mean2, bar_width, alpha=opacity, color='blue',  yerr=sem2,capsize=10,ecolor='black',label='Template')

# Scatter the data points for both groups along the x-axis
plt.scatter(np.repeat(x_wt, len(group1_data)) +np.random.uniform(-jitter,jitter,len(group1_data)), group1_data, marker='o',s=5,color='green')
plt.scatter(np.repeat(x_het, len(group2_data))+np.random.uniform(-jitter,jitter,len(group2_data)), group2_data, marker='o',s=5,color='orange')

# Create bars for Group 1 with two colors
plt.bar(x_wt+0.15, mean3, bar_width, alpha=opacity, color='red', yerr=sem3, capsize=10,ecolor='black')
plt.bar(x_het+0.15, mean4, bar_width, alpha=opacity, color='blue',  yerr=sem4,capsize=10,ecolor='black')

# Scatter the data points for both groups along the x-axis
plt.scatter(np.repeat(x_wt+0.15, len(group3_data)) +np.random.uniform(-jitter,jitter,len(group3_data)), group3_data, marker='o',s=5,color='green')
plt.scatter(np.repeat(x_het+0.15, len(group4_data))+np.random.uniform(-jitter,jitter,len(group4_data)), group4_data, marker='o',s=5,color='orange')

# Customize the plot

plt.ylabel('PCC scores')
plt.title('PCC scores of individual bursts from control and Syngap comparison with control template')
plt.legend()
plt.xticks([1.0225,1.175],['DIV 88','DIV 101'])  # Remove x-axis ticks
plt.ylim([0.4,1.25])

plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
# Show the plot

plt.savefig("/home/mmp/Documents/Images_25sep/templatematching_plot.pdf",format='pdf' ,bbox_inches='tight')


In [ ]:
print(mean1, sem1)
print(mean2, sem2)
print(mean3, sem3)
print(mean4, sem4)

In [ ]:
template_list =[]
for i in range(len(extracted_templates)):
    values_str = extracted_templates.iloc[i].strip('[]').split()
    # Convert the string values to floats
    values_float = [float(val) for val in values_str]
    template_list.append(values_float)

template_array = np.array(template_list)

mean_template = np.mean(template_array,axis=0)

print(mean_template)

In [ ]:
plt.figure(figsize=(2, 10)) 
plt.plot(mean_template)

plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Stacked Bursts')

plt.show()

In [ ]:
extracted_bursts = filtered_df['Stacked Bursts'].str.replace('\n', '')

In [ ]:
extracted_bursts.iloc[0]

In [ ]:
extracted_bursts.iloc[0].split(']')

    

In [ ]:
import ast

# Your string containing the list of lists as a string
input_str = '[[0.47613212 0.50067745 0.52616663 1.5886198  1.52691248 1.46659206] [0.59402887 0.61821148 0.64392679 1.74584332 1.68953781 1.63262709] [0.6686465  0.68204741 0.6972058 1.54553606 1.48494743 1.42560207]]'

# Add commas between the elements within the inner lists
input_str = input_str.replace(']', '],')
input_str =  input_str[:-3]+']'  # Remove the trailing comma

# Use ast.literal_eval to safely evaluate the string as a list of lists
list_of_lists = ast.literal_eval(input_str)

print(list_of_lists)

In [ ]:
import re



# Use regular expressions to extract the numeric values
numeric_values = re.findall(r'[-+]?\d*\.\d+|\d+', extracted_bursts.iloc[0])

# Convert the extracted numeric values to a list of lists of floats
list_of_lists_float = []

# Specify the number of columns in each sublist
num_columns = 10  # Adjust this number as per your data

# Split the numeric values into sublists
for i in range(0, len(numeric_values), num_columns):
    sublist = [float(val) for val in numeric_values[i:i+num_columns]]
    list_of_lists_float.append(sublist)

In [ ]:
list_of_lists_float

In [ ]:
 #Convert the string format to a list of lists with float values
list_of_lists_float = [[float(val) for val in eval(inner_list)] for inner_list in extracted_bursts.iloc[0]]
list_of_lists_float

In [ ]:
for i in range(len(extracted_bursts)):
    

In [ ]:
values_str = extracted_templates.iloc[0].strip('[]').split()

# Convert the string values to floats
values_float = [float(val) for val in values_str]

In [ ]:
values_float

In [ ]:
import ast
val = ast.literal_eval(extracted_templates.iloc[2])
val

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt


def calculate_burst_features(stacked_bursts, threshold=0.5):

    feature_array = np.empty((len(stacked_bursts), 1)) 
    
    for i,burst in enumerate(stacked_bursts):
    
        # Find the peak amplitude and its index
        peak_amplitude = np.max(burst)
        peak_index = np.argmax(burst)
        # Calculate the half-maximum threshold
        half_threshold = threshold * peak_amplitude

        # Search for the left boundary where the amplitude first crosses half-maximum
        left_boundary = np.where(burst[:peak_index] <= half_threshold)[0]
        if len(left_boundary) == 0:
            left_boundary_index = 0
        else:
            left_boundary_index = left_boundary[-1] + 1

        # Search for the right boundary where the amplitude crosses half-maximum again
        right_boundary = np.where(burst[peak_index:] <= half_threshold)[0]
        if len(right_boundary) == 0:
            right_boundary_index = len(burst) - 1
        else:
            right_boundary_index = peak_index + right_boundary[0]

        # Calculate the half-width
        half_width = right_boundary_index - left_boundary_index + 1

        feature_array[i] = [peak_amplitude]
    

    return feature_array

# Example usage:

features = calculate_burst_features(stacked_bursts, threshold=0.5)











In [ ]:
print(features)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Assuming features is a NumPy array where each row represents a burst with two columns (half-width, peak)

# Initialize an empty list to store the WCSS values for different values of K
wcss = []

# Define a range of values for K (number of clusters)
k_values = range(1, 11)  # You can adjust the range as needed

# Calculate WCSS for each value of K
for k in k_values:
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(features[:,0].reshape(-1,1))
    wcss.append(kmeans.inertia_)
# Calculate the change in slope between consecutive WCSS values
slope_changes = [abs(wcss[i] - wcss[i - 1] )for i in range(1, len(wcss))]

# Find the index with the maximum slope change
optimal_k_index = slope_changes.index(max(slope_changes)) + 1  # Adding 1 because of zero-based indexing

# Optimal K is the K value at the inflection point with the maximum slope change
optimal_k = k_values[optimal_k_index]
# Plot the WCSS values to determine the elbow point
plt.figure(figsize=(8, 5))
plt.plot(k_values, wcss, marker='o', linestyle='-', color='b')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.title('Elbow Method for Optimal K')
plt.grid(True)
plt.show()


In [ ]:
print(optimal_k)

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Assuming features is a NumPy array where each row represents a burst with two columns (half-width, peak)
# Assuming you have already determined the optimal number of clusters, e.g., optimal_k

# Initialize and fit K-Means with the optimal number of clusters
kmeans = KMeans(n_clusters=optimal_k)
cluster_labels = kmeans.fit_predict(features)
print(cluster_labels)
# Now, cluster_labels contains the cluster assignments for each burst

# Create a dictionary to store features and their indices for each cluster
cluster_data = {cluster_id: {'features': [], 'indices': []} for cluster_id in range(optimal_k)}

# Assign features and their indices to their respective clusters
for i, label in enumerate(cluster_labels):
    cluster_data[label]['features'].append(features[i])
    cluster_data[label]['indices'].append(i)

# Access features and indices for a specific cluster (e.g., cluster 0)
cluster_0_features = np.array(cluster_data[0]['features'])
cluster_0_indices = np.array(cluster_data[0]['indices'])

cluster_1_indices = np.array(cluster_data[1]['indices'])


In [ ]:
plt.figure(figsize=(2, 12))  # Adjust the figure size as needed

for row in stacked_bursts[cluster_0_indices,:]:
    plt.plot(row, c='red')
for row in stacked_bursts[cluster_1_indices,:]:
    plt.plot(row, c='blue')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Stacked Bursts')
plt.grid(True)
plt.show()
    

    

In [ ]:
template1 = np.mean(stacked_bursts[cluster_0_indices,:],axis =0)
template2 = np.mean(stacked_bursts[cluster_1_indices,:], axis =0)
plt.figure(figsize=(2, 12))
plt.plot(template2,c='blue')
plt.plot(template1,c='red')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Stacked Bursts')
plt.grid(True)
plt.show()

In [ ]:
print(stacked_bursts[1,:].shape)

In [ ]:
template= np.mean(stacked_bursts,axis=0)

In [ ]:
template.shape

In [ ]:
plt.figure(figsize=(2,10))
plt.plot(stacked_bursts[:,1])

plt.xlabel('Time')
plt.ylabel('kHz')
plt.title('Line Plot of Data')
plt.show()

In [ ]:
#generating the template, which is the average among the bursts
num_burst = len(list_of_burst)
point_on_burst = len(list_of_burst[0])
sum_of_burst = [0]*point_on_burst
for burst in list_of_burst:
    for x in range(point_on_burst):
        sum_of_burst[x] += burst[x]
    
template= [total / num_burst for total in sum_of_burst]

print(sum_of_burst)

In [ ]:
print(template)
plt.plot(template)
plt.xlabel('Time')
plt.ylabel('kHz')
plt.title('Line Plot of Data')
plt.show()

In [ ]:
#find the pearson correlation coefficient between each individual burst and the template
pcc=[]
for burst in stacked_bursts:
    pcc.append(np.corrcoef(burst, template)[0, 1])

print(pcc)
len(pcc)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans



# Reshape the data to fit the KMeans input format
X = np.array(pcc).reshape(-1, 1)
distortions =[]
for i in range(1,6):
    # Create a KMeans instance with 4 clusters
    kmeans = KMeans(n_clusters=i)

    # Fit the model to the data
    kmeans.fit(X)
    distortions.append(kmeans.inertia_)

plt.plot([1,2,3,4,5],distortions)





In [ ]:
kmeans = KMeans(n_clusters=2)

# Fit the model to the data
kmeans.fit(X)
# Get cluster labels for each data point
cluster_labels = kmeans.labels_

# Get the centroids of each cluster
centroids = kmeans.cluster_centers_

print("Cluster Labels:", cluster_labels)
print("Centroids:", centroids)

In [ ]:
#find the average PCC
sum = 0
for value in pcc:
    sum+=value
average_pcc= sum/(len(pcc))

print(average_pcc)